# 💰 Task 4: Sales Prediction using Python
**Dataset:** advertising.csv — 200 records of TV, Radio, Newspaper ad spend vs Sales

Columns: `TV`, `Radio`, `Newspaper`, `Sales`

In [ ]:
# !pip install scikit-learn pandas matplotlib seaborn

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded!")

## 2. Load the Real Dataset

In [ ]:
# Load directly from GitHub
url = "https://raw.githubusercontent.com/amankharwal/Website-data/master/advertising.csv"
df = pd.read_csv(url)

# OR local: df = pd.read_csv("advertising.csv")

print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
print("Dataset Info:")
df.info()
print("\nMissing values:", df.isnull().sum().sum())
print("\nStatistical Summary:")
df.describe()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Histograms
for i, col in enumerate(['TV', 'Radio', 'Newspaper', 'Sales']):
    ax = axes[i // 3][i % 3] if i < 3 else axes[1][i-3]
    ax.hist(df[col], bins=20, color=['#3498DB','#E74C3C','#2ECC71','#9B59B6'][i], edgecolor='white')
    ax.set_title(f'{col} Distribution', fontweight='bold')
    ax.set_xlabel(col)

# Scatter: TV vs Sales
axes[1][1].scatter(df['TV'], df['Sales'], alpha=0.6, color='#3498DB', s=30)
axes[1][1].set_xlabel('TV Ad Spend ($000s)'); axes[1][1].set_ylabel('Sales ($000s)')
axes[1][1].set_title('TV Spend vs Sales', fontweight='bold')

# Scatter: Radio vs Sales
axes[1][2].scatter(df['Radio'], df['Sales'], alpha=0.6, color='#E74C3C', s=30)
axes[1][2].set_xlabel('Radio Ad Spend ($000s)'); axes[1][2].set_ylabel('Sales ($000s)')
axes[1][2].set_title('Radio Spend vs Sales', fontweight='bold')

plt.suptitle('Advertising Dataset EDA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sales_eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Pairplot
sns.pairplot(df, diag_kind='kde', plot_kws={'alpha':0.5, 's':20},
             vars=['TV','Radio','Newspaper','Sales'])
plt.suptitle('Pairplot — Advertising Dataset', y=1.02, fontsize=13, fontweight='bold')
plt.savefig('sales_pairplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(), annot=True, fmt='.3f', cmap='coolwarm', square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('sales_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Correlation with Sales:")
print(df.corr()['Sales'].sort_values(ascending=False))

## 4. Feature Engineering

In [ ]:
df_ml = df.copy()

# Interaction features
df_ml['TV_Radio']       = df_ml['TV'] * df_ml['Radio']
df_ml['TV_Newspaper']   = df_ml['TV'] * df_ml['Newspaper']
df_ml['Total_Spend']    = df_ml['TV'] + df_ml['Radio'] + df_ml['Newspaper']
df_ml['TV_Share']       = df_ml['TV'] / (df_ml['Total_Spend'] + 1)
df_ml['Radio_Share']    = df_ml['Radio'] / (df_ml['Total_Spend'] + 1)

features_basic    = ['TV', 'Radio', 'Newspaper']
features_extended = ['TV', 'Radio', 'Newspaper', 'TV_Radio', 'TV_Newspaper', 'Total_Spend', 'TV_Share', 'Radio_Share']

print("Basic features    :", features_basic)
print("Extended features :", features_extended)
df_ml.head()

## 5. Train/Test Split

In [ ]:
X = df_ml[features_extended]
y = df_ml['Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

## 6. Train Models

In [ ]:
models = {
    'Linear Regression':  LinearRegression(),
    'Ridge Regression':   Ridge(alpha=1.0),
    'Lasso Regression':   Lasso(alpha=0.01),
    'Decision Tree':      DecisionTreeRegressor(max_depth=6, random_state=42),
    'Random Forest':      RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    mae    = mean_absolute_error(y_test, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    r2     = r2_score(y_test, y_pred)
    cv     = cross_val_score(model, X_train_s, y_train, cv=5, scoring='r2')
    results[name] = {'R2': r2, 'MAE': mae, 'RMSE': rmse, 'CV_R2': cv.mean()}
    print(f"{name:<22} R²={r2:.4f}  MAE={mae:.3f}  RMSE={rmse:.3f}  CV_R²={cv.mean():.4f}")

## 7. Model Comparison

In [ ]:
res_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)
clrs   = plt.cm.plasma(np.linspace(0.15, 0.85, len(res_df)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bars = axes[0].bar(res_df.index, res_df['R2'], color=clrs)
axes[0].set_ylabel('R² Score'); axes[0].set_ylim(0, 1.1)
axes[0].set_title('R² Score — All Models', fontweight='bold')
axes[0].set_xticklabels(res_df.index, rotation=20, ha='right')
for bar, v in zip(bars, res_df['R2']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{v:.3f}', ha='center', fontweight='bold', fontsize=9)

axes[1].bar(res_df.index, res_df['MAE'], color=clrs)
axes[1].set_ylabel('MAE (Sales $000s)'); axes[1].set_title('Mean Absolute Error', fontweight='bold')
axes[1].set_xticklabels(res_df.index, rotation=20, ha='right')
plt.tight_layout()
plt.savefig('sales_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Best Model — Actual vs Predicted & Residuals

In [ ]:
best_name  = res_df.index[0]
best_model = models[best_name]
y_pred     = best_model.predict(X_test_s)
residuals  = y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, y_pred, alpha=0.7, color='#3498DB', s=40, edgecolors='white')
lims = [min(y_test.min(), y_pred.min())-0.5, max(y_test.max(), y_pred.max())+0.5]
axes[0].plot(lims, lims, 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Sales'); axes[0].set_ylabel('Predicted Sales')
axes[0].set_title(f'{best_name} — Actual vs Predicted', fontweight='bold')
axes[0].legend()

axes[1].scatter(y_pred, residuals, alpha=0.7, color='#E74C3C', s=40, edgecolors='white')
axes[1].axhline(0, color='black', lw=1.5)
axes[1].set_xlabel('Predicted Sales'); axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot', fontweight='bold')
plt.tight_layout()
plt.savefig('sales_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Best Model : {best_name}")
print(f"R² Score   : {results[best_name]['R2']:.4f}")
print(f"MAE        : {results[best_name]['MAE']:.3f} ($000s)")
print(f"RMSE       : {results[best_name]['RMSE']:.3f} ($000s)")

## 9. Feature Importance & ROI Analysis

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=features_extended).sort_values(ascending=True)
    plt.figure(figsize=(9, 5))
    imp.plot(kind='barh', color='#3498DB')
    plt.title(f'Feature Importance — {best_name}', fontsize=13, fontweight='bold')
    plt.xlabel('Importance Score')
    plt.tight_layout()
    plt.savefig('sales_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

# ROI per $ spent on each channel
print("\nReturn on Ad Spend (correlation with Sales):")
for col in ['TV','Radio','Newspaper']:
    corr = df[col].corr(df['Sales'])
    print(f"  {col:<12}: r = {corr:.4f}")

print()
print("Avg spend per channel (in $000s):")
print(df[['TV','Radio','Newspaper']].mean().round(2))

## 10. Predict Future Sales

In [ ]:
# Predict for 5 different advertising budgets
scenarios = pd.DataFrame({
    'TV':        [50, 100, 150, 200, 250],
    'Radio':     [10,  20,  30,  40,  50],
    'Newspaper': [ 5,  10,  15,  20,  25],
})
# Add engineered features
scenarios['TV_Radio']     = scenarios['TV'] * scenarios['Radio']
scenarios['TV_Newspaper'] = scenarios['TV'] * scenarios['Newspaper']
scenarios['Total_Spend']  = scenarios['TV'] + scenarios['Radio'] + scenarios['Newspaper']
scenarios['TV_Share']     = scenarios['TV'] / (scenarios['Total_Spend'] + 1)
scenarios['Radio_Share']  = scenarios['Radio'] / (scenarios['Total_Spend'] + 1)

pred_sales = best_model.predict(scaler.transform(scenarios[features_extended]))
scenarios['Predicted_Sales'] = pred_sales.round(2)

print("Sales Forecast for Different Ad Budgets:")
print(scenarios[['TV','Radio','Newspaper','Predicted_Sales']].to_string(index=False))

plt.figure(figsize=(9, 4))
plt.plot(scenarios['Total_Spend'], scenarios['Predicted_Sales'], 'bo-', lw=2, markersize=8)
plt.xlabel('Total Ad Spend ($000s)'); plt.ylabel('Predicted Sales ($000s)')
plt.title('Sales Forecast vs Total Ad Spend', fontsize=13, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('sales_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Business Marketing Insights

In [ ]:
print("=" * 55)
print("ACTIONABLE MARKETING INSIGHTS")
print("=" * 55)
tv_corr   = df['TV'].corr(df['Sales'])
rad_corr  = df['Radio'].corr(df['Sales'])
news_corr = df['Newspaper'].corr(df['Sales'])

print(f"1. TV Advertising     — correlation with Sales: {tv_corr:.3f}  ← Strongest driver")
print(f"2. Radio Advertising  — correlation with Sales: {rad_corr:.3f}  ← Second best")
print(f"3. Newspaper Ads      — correlation with Sales: {news_corr:.3f}  ← Weakest ROI")
print()
print(f"Best Prediction Model : {best_name}  (R² = {results[best_name]['R2']:.4f})")
print()
print("RECOMMENDATIONS:")
print("  • Allocate highest budget to TV — it has the strongest sales correlation")
print("  • Radio is cost-effective — high ROI for smaller budgets")
print("  • Reduce Newspaper budget — lowest correlation with sales")
print("  • Combined TV + Radio campaigns show synergy effects")
print("  • Use this model to simulate budget allocation scenarios")